# Fase 2e: detector de roya para no confundirla con ojo de gallo
Tesis: aplicación móvil para identificar síntomas de ojo de gallo (Rodrigo Alfaro, UCC).

**Problema.** En el teléfono, fotos de hojas con roya salen como "síntomas compatibles con ojo de gallo, certeza alta"
(P(ojo) entre 0,96 y 0,997). La causa: el diagnóstico v1 se entrenó solo con ojo de gallo y sanas; nunca vio roya, así
que aprendió "hoja con manchas" y no "ojo de gallo".

**Propuesta.** Un segundo modelo, independiente, que responde una sola pregunta: **¿esta hoja tiene roya?**
Se entrena con roya de **tres orígenes** (Perú: Mendeley; Ecuador: RoCoLe, en campo; Brasil: BRACOL) y, como
ejemplos de "no roya", hojas sanas de los tres orígenes, ojo de gallo (Mendeley) y otras manchas de BRACOL (minador,
phoma, cercospora). Así el modelo no puede usar el origen de la foto como atajo: hay roya y no roya en cada origen.

**Evaluación sin trampa.** PG 26038 (Costa Rica, 40 ojo de gallo, 40 roya, 20 sanas) **no se usa para entrenar ni
para elegir umbrales**. El umbral se fija con la validación. También se compara con una regla simple de color
(proporción de píxeles anaranjados), para saber si el modelo aporta algo más que el color.

**Cómo usarlo:** Entorno de ejecución → Cambiar tipo de entorno → T4 GPU. Luego Entorno de ejecución → Ejecutar todas
(unos 25 minutos; la mitad es descomprimir RoCoLe).

In [1]:
import os, sys, json, math, time, random, glob, platform, subprocess
MOCK = os.environ.get("MOCK") == "1"
if not MOCK:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "importlib_resources", "openpyxl"], check=False)
    from google.colab import drive
    drive.mount("/content/drive")
import numpy as np, pandas as pd, tensorflow as tf, keras
from PIL import Image, ImageOps, ImageFile
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import roc_auc_score, roc_curve
ImageFile.LOAD_TRUNCATED_IMAGES = False

SEMILLA = 42; TAM = 224
EPOCAS = int(os.environ.get("EPOCAS", 20))
BASE = os.environ.get("RUTA_BASE", "/content/drive/MyDrive/tesis_ojo_de_gallo")
DS = os.path.join(BASE, "02_Dataset")
TMP = "/content/tmp_fase2e" if not MOCK else "/tmp/tmp_fase2e"
RUTAS = {
    "ojo_de_gallo": os.path.join(DS, "Ojo_Gallo"), "sana": os.path.join(DS, "Sanas"), "roya": os.path.join(DS, "Roya"),
    "externos": os.path.join(DS, "externos"),
    "bracol_imagenes": os.path.join(DS, "externos", "BRACOL", "leaf", "images"),
    "pg26038": os.path.join(DS, "externos", "PG26038_muestras", "images"),
    "diag_v1": os.path.join(BASE, "fase2_resultados", "android", "modelo_diagnostico.tflite"),
    "ver_v3": os.path.join(BASE, "fase2c_modelos_v3", "android", "modelo_verificador.tflite"),
}
SAL = os.path.join(BASE, "fase2e_roya")
for sub in ["tablas", "android"]:
    os.makedirs(os.path.join(SAL, sub), exist_ok=True)
os.makedirs(TMP, exist_ok=True)
EXT = (".jpg", ".jpeg", ".png")

def fijar_semillas(s=SEMILLA):
    random.seed(s); np.random.seed(s); keras.utils.set_random_seed(s)
fijar_semillas()
VERSIONES = {"python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__,
             "numpy": np.__version__, "gpu": [d.name for d in tf.config.list_physical_devices("GPU")]}
print(VERSIONES)
for k, r in RUTAS.items():
    print(("OK   " if os.path.exists(r) else "FALTA"), k, r)
def guardar_json(obj, nombre):
    with open(os.path.join(SAL, "tablas", nombre), "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=float)
def wilson(k, n, z=1.959963984540054):
    if n == 0: return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

Mounted at /content/drive
{'python': '3.13.16', 'tensorflow': '2.21.0', 'keras': '3.13.2', 'numpy': '2.1.3', 'gpu': ['/physical_device:GPU:0']}
OK    ojo_de_gallo /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/Ojo_Gallo
OK    sana /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/Sanas
OK    roya /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/Roya
OK    externos /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/externos
OK    bracol_imagenes /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/externos/BRACOL/leaf/images
OK    pg26038 /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/externos/PG26038_muestras/images
OK    diag_v1 /content/drive/MyDrive/tesis_ojo_de_gallo/fase2_resultados/android/modelo_diagnostico.tflite
OK    ver_v3 /content/drive/MyDrive/tesis_ojo_de_gallo/fase2c_modelos_v3/android/modelo_verificador.tflite


## 1. Lectura de imágenes (igual que la aplicación)
Orientación EXIF corregida, RGB (PNG transparentes sobre blanco), redimensión bilineal a 224 × 224 con
`tf.image.resize`. Se guardan como enteros de 0 a 255 para ahorrar memoria.

In [2]:
DANADAS = []
def leer(ruta):
    with Image.open(ruta) as im:
        im = ImageOps.exif_transpose(im)
        if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
            im = Image.alpha_composite(Image.new("RGBA", im.size, (255, 255, 255, 255)), im.convert("RGBA"))
        arr = np.asarray(im.convert("RGB"), dtype=np.float32)
    return np.clip(np.round(tf.image.resize(arr[None], (TAM, TAM), method="bilinear").numpy()[0]), 0, 255).astype("uint8")
def leer_varias(rutas, desc):
    t0 = time.time()
    def seguro(r):
        try: return leer(r)
        except Exception as e: DANADAS.append({"imagen": r, "error": str(e)}); return None
    with ThreadPoolExecutor(8) as ex: res = list(ex.map(seguro, rutas))
    ok = [r for r, x in zip(rutas, res) if x is not None]
    print(f"{desc}: {len(ok)} de {len(rutas)} en {time.time() - t0:.0f} s")
    X = np.stack([x for x in res if x is not None]) if ok else np.zeros((0, TAM, TAM, 3), "uint8")
    return X, ok
def listar(c, recursivo=False):
    if not os.path.isdir(c): return []
    if recursivo:
        return sorted(os.path.join(r, f) for r, _, fs in os.walk(c) for f in fs if f.lower().endswith(EXT))
    return sorted(os.path.join(c, f) for f in os.listdir(c) if f.lower().endswith(EXT))
def dividir(n, semilla=SEMILLA):
    idx = np.random.RandomState(semilla).permutation(n)
    a = int(round(0.70 * n)); b = a + int(round(0.15 * n))
    return {"train": idx[:a], "validation": idx[a:b], "test": idx[b:]}
def ds_desde(X, y, entrenar):
    d = tf.data.Dataset.from_tensor_slices((X, y))
    if entrenar: d = d.shuffle(len(X), seed=SEMILLA, reshuffle_each_iteration=True)
    return d.batch(32).map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

## 2. RoCoLe (Ecuador, fotos en campo) con sus etiquetas
RoCoLe (Parraga-Alava et al., 2019; DOI 10.17632/c5yvn32dzg.2; CC BY 4.0). Las etiquetas vienen en la carpeta
`Annotations` (sana, ácaro rojo y roya de nivel 1 a 4). Se usa el ZIP que está en `02_Dataset/externos/`.

In [3]:
def descomprimir(zp, destino):
    os.makedirs(destino, exist_ok=True)
    r = subprocess.run(["7z", "x", "-y", "-o" + destino, zp], capture_output=True, text=True)
    if r.returncode != 0:
        print("7z:", r.stdout[-500:], r.stderr[-500:])

RO_DIR = os.path.join(TMP, "rocole")
if not listar(RO_DIR, True) and not MOCK:
    zips = [z for z in glob.glob(os.path.join(RUTAS["externos"], "*.zip")) if "rocole" in os.path.basename(z).lower()]
    print("ZIP de RoCoLe:", zips)
    for z in zips:
        t0 = time.time(); descomprimir(z, RO_DIR); print(f"Descomprimido en {time.time() - t0:.0f} s")
    for z in glob.glob(os.path.join(RO_DIR, "**", "*.zip"), recursive=True):     # ZIP dentro del ZIP
        descomprimir(z, os.path.dirname(z))
if MOCK:
    RO_DIR = os.path.join(RUTAS["externos"], "RoCoLe")
fotos_rocole = {os.path.basename(r): r for r in listar(RO_DIR, True)}
print("Fotos de RoCoLe:", len(fotos_rocole))
assert fotos_rocole, "No se encontró RoCoLe. Sube 'RoCoLe A robusta coffee leaf images dataset.zip' a 02_Dataset/externos."

def tabla_anotaciones(carpeta):
    """Busca la tabla de etiquetas de RoCoLe (xlsx o csv) y devuelve (archivo, etiqueta) normalizados."""
    candidatos = [f for f in glob.glob(os.path.join(carpeta, "**", "*"), recursive=True)
                  if f.lower().endswith((".xlsx", ".csv")) and not os.path.basename(f).startswith("~")]
    for f in candidatos:
        try:
            df = pd.read_excel(f) if f.lower().endswith(".xlsx") else pd.read_csv(f)
        except Exception as e:
            print("No se pudo leer", f, e); continue
        cols = {c.lower(): c for c in df.columns}
        c_arch = next((cols[c] for c in cols if "file" in c or "archivo" in c or "image" in c), None)
        c_multi = next((cols[c] for c in cols if "multi" in c), None)
        if c_arch and c_multi:
            print("Etiquetas de RoCoLe:", os.path.relpath(f, carpeta), "| columnas:", list(df.columns))
            return pd.DataFrame({"archivo": df[c_arch].astype(str).map(os.path.basename),
                                 "etiqueta": df[c_multi].astype(str).str.lower().str.strip()})
    print("Archivos revisados:", [os.path.relpath(f, carpeta) for f in candidatos])
    raise RuntimeError("No se encontró la tabla de etiquetas de RoCoLe (columna 'Multiclass').")

ANOT = tabla_anotaciones(RO_DIR)
print(ANOT["etiqueta"].value_counts())
def clase_rocole(etq):
    if "rust" in etq: return "roya"
    if "healthy" in etq: return "sana"
    return "otra"                                  # ácaro rojo: no es roya
ANOT["clase"] = ANOT["etiqueta"].map(clase_rocole)
ANOT["ruta"] = ANOT["archivo"].map(fotos_rocole)
sin_foto = ANOT["ruta"].isna().sum()
print("Etiquetas sin foto:", sin_foto)
ANOT = ANOT.dropna(subset=["ruta"]).drop_duplicates("archivo").reset_index(drop=True)
print(ANOT["clase"].value_counts())

ZIP de RoCoLe: ['/content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/externos/RoCoLe A robusta coffee leaf images dataset.zip']
Descomprimido en 55 s
Fotos de RoCoLe: 1560
Etiquetas de RoCoLe: RoCoLe A robusta coffee leaf images dataset/Annotations/RoCoLe-classes.xlsx | columnas: ['File', 'Binary.Label', 'Multiclass.Label']
etiqueta
healthy            791
rust_level_1       344
red_spider_mite    167
rust_level_2       166
rust_level_3        62
rust_level_4        30
Name: count, dtype: int64
Etiquetas sin foto: 0
clase
sana    791
roya    602
otra    167
Name: count, dtype: int64


## 3. BRACOL (Brasil) con sus etiquetas
BRACOL (Krohling, Esgario y Ventura, 2019; DOI 10.17632/yy2k5y8mxg.1; CC BY 4.0). La tabla `dataset.csv` indica qué
estrés tiene cada hoja (minador, roya, phoma, cercospora). Roya = columna `rust` mayor que 0.

In [4]:
tablas_b = glob.glob(os.path.join(RUTAS["externos"], "BRACOL", "**", "dataset.csv"), recursive=True)
print("Tablas de BRACOL:", tablas_b)
assert tablas_b, "No se encontró BRACOL/leaf/dataset.csv en 02_Dataset/externos."
TB = pd.read_csv(tablas_b[0]); print("Columnas:", list(TB.columns)); print(TB.head(3))
cols = {c.lower(): c for c in TB.columns}
c_id = cols.get("id", TB.columns[0])
if "rust" in cols:
    es_roya = TB[cols["rust"]].astype(float) > 0
else:
    es_roya = TB[cols["predominant_stress"]].astype(int) == 2      # 2 = roya según la documentación de BRACOL
fotos_b = {os.path.splitext(os.path.basename(r))[0]: r for r in listar(RUTAS["bracol_imagenes"])}
TB["ruta"] = TB[c_id].astype(str).map(lambda i: fotos_b.get(os.path.splitext(i)[0]))
TB["clase"] = np.where(es_roya, "roya", "no_roya")
TB = TB.dropna(subset=["ruta"]).reset_index(drop=True)
print(TB["clase"].value_counts())

Tablas de BRACOL: ['/content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/externos/BRACOL/leaf/dataset.csv']
Columnas: ['id', 'predominant_stress', 'miner', 'rust', 'phoma', 'cercospora', 'severity']
   id  predominant_stress  miner  rust  phoma  cercospora  severity
0   1                   0      0     0      0           0         0
1   2                   0      0     0      0           0         0
2   3                   0      0     0      0           0         0
clase
no_roya    797
roya       605
Name: count, dtype: int64


## 4. Mendeley (Perú) y PG 26038 (Costa Rica, solo prueba)
Mendeley: misma partición de las fases anteriores (ojo de gallo y sanas de la Fase 2; roya 70/15/15, semilla 42).

In [5]:
PARTICION = json.loads(r'''{"train": {"ojo_de_gallo": ["20251129_091959.jpg", "20251129_092016.jpg", "20251129_092053.jpg", "20251129_092106.jpg", "20251129_092123.jpg", "20251129_092139.jpg", "20251129_092149.jpg", "20251129_092158.jpg", "20251129_092220.jpg", "20251129_092252.jpg", "20251129_092438.jpg", "20251129_092457.jpg", "20251129_092504.jpg", "20251129_092511.jpg", "20251129_092539.jpg", "20251129_092559.jpg", "20251129_092702.jpg", "20251129_092708.jpg", "20251129_092800.jpg", "20251129_092931.jpg", "20251129_093055.jpg", "20251129_093132.jpg", "20251129_093207.jpg", "20251129_093218.jpg", "20251129_093257.jpg", "20251129_093304.jpg", "20251129_093324.jpg", "20251129_093359.jpg", "20251129_093502.jpg", "20251129_093615.jpg", "20251129_093623.jpg", "20251129_093725.jpg", "20251129_093729.jpg", "20251129_093736.jpg", "20251129_093956.jpg", "20251129_094006.jpg", "20251129_094032.jpg", "20251129_094102.jpg", "20251129_094107.jpg", "20251129_094130.jpg", "20251129_094219.jpg", "20251129_094307.jpg", "20251129_094337.jpg", "20251129_094614.jpg", "20251129_094622.jpg", "20251129_094627.jpg", "20251129_094654.jpg", "20251129_094701.jpg", "20251129_094834.jpg", "20251129_094950.jpg", "20251129_095011.jpg", "20251129_095106.jpg", "20251129_095132.jpg", "20251129_095138.jpg", "20251129_095206.jpg", "20251129_095244.jpg", "20251129_095331.jpg", "20251129_095408.jpg", "20251129_095416.jpg", "20251129_095449.jpg", "20251129_095512.jpg", "20251129_095538.jpg", "20251129_095552.jpg", "20251129_095556.jpg", "20251129_095601.jpg", "20251129_095609.jpg", "20251129_095629.jpg", "20251129_095642.jpg", "20251129_095712.jpg", "20251129_095729.jpg", "20251129_095736.jpg", "20251129_095755.jpg", "20251129_095810.jpg", "20251129_095821.jpg", "20251129_100046.jpg", "20251129_100416.jpg", "20251129_100436.jpg", "20251129_100506.jpg", "20251129_100515.jpg", "20251129_100535.jpg", "20251129_100547.jpg", "20251129_100621.jpg", "20251129_100711.jpg", "20251129_100808.jpg", "20251129_100906.jpg", "20251129_100918.jpg", "20251129_100948.jpg", "20251129_100955.jpg", "20251129_101023.jpg", "20251129_101029.jpg", "20251129_101033.jpg", "20251129_101157.jpg", "20251129_101255.jpg", "20251129_101307.jpg", "20251129_101319.jpg", "20251129_101337.jpg", "20251129_101342.jpg", "20251129_101430.jpg", "20251129_101441.jpg", "20251129_101504.jpg", "20251129_101509.jpg", "20251129_101518.jpg", "20251129_101529.jpg", "20251129_101559.jpg", "20251129_101610.jpg", "20251129_101624.jpg", "20251129_101641.jpg", "20251129_101714.jpg", "20251129_101831.jpg", "20251129_101839.jpg", "20251129_101856.jpg", "20251129_101905.jpg", "20251129_102019.jpg", "20251129_102125.jpg", "20251129_102402.jpg", "20251129_102518.jpg", "20251129_102657.jpg", "20251129_102737.jpg", "20251129_102838.jpg", "20251129_102905.jpg", "20251129_103003.jpg", "20251129_103051.jpg", "20251129_103124.jpg", "20251129_103139.jpg", "20251129_103154.jpg", "20251129_103206.jpg", "20251129_103304.jpg", "20251129_103311.jpg", "20251129_103353.jpg", "20251129_103409.jpg", "20251129_103437.jpg", "20251129_103445.jpg", "20251129_103532.jpg", "20251129_103542.jpg", "20251129_103604.jpg", "20251129_103618.jpg", "20251129_103647.jpg", "20251129_103728.jpg", "20251129_103758.jpg", "20251129_103804.jpg", "20251129_103839.jpg", "20251129_103847.jpg", "20251129_103938.jpg", "20251129_104212.jpg", "20251129_104314.jpg", "20251129_104320.jpg", "20251129_104346.jpg", "20251129_104351.jpg", "20251129_104356.jpg", "20251129_104409.jpg", "20251129_104557.jpg", "20251129_104706.jpg", "20251129_104727.jpg", "20251129_104745.jpg", "20251129_104803.jpg", "20251129_104901.jpg", "20251129_104924.jpg", "20251129_104958.jpg", "20251129_105257.jpg", "20251129_105404.jpg", "20251129_105435.jpg", "20251129_105505.jpg", "20251129_105517.jpg", "20251129_105537.jpg", "20251129_114546.jpg", "20251129_114556.jpg", "20251129_114630.jpg", "20251129_114655.jpg", "20251129_114830.jpg", "20251129_114858.jpg", "20251129_114932.jpg", "20251129_114957.jpg", "20251129_115010.jpg", "20251129_115017.jpg", "20251129_115123.jpg", "20251129_115140.jpg", "20251129_115201.jpg", "20251129_115218.jpg", "20251129_115223.jpg", "20251129_115251.jpg", "20251129_115305.jpg", "20251129_115313.jpg", "20251129_115322.jpg", "20251129_115331.jpg", "20251129_115426.jpg", "20251129_115503.jpg", "20251129_115534.jpg", "20251129_115633.jpg", "20251129_115719.jpg", "20251129_115734.jpg", "20251129_115754.jpg", "20251129_115815.jpg", "20251129_115822.jpg", "20251129_115827.jpg", "20251129_115900.jpg", "20251129_115923.jpg", "20251129_120045.jpg", "20251129_120152.jpg", "20251129_120217.jpg", "20251129_120227.jpg", "20251129_120259.jpg", "20251129_120417.jpg", "20251129_120512.jpg", "20251129_120518.jpg", "20251129_120531.jpg", "20251129_120544.jpg", "20251129_120552.jpg", "20251129_120745.jpg", "20251129_120748.jpg", "20251129_120803.jpg", "20251129_120820.jpg", "20251129_120831.jpg", "20251129_120901.jpg", "20251129_120915.jpg", "20251129_120956.jpg", "20251129_121017.jpg", "20251129_121030.jpg", "20251129_121051.jpg", "20251129_121134.jpg", "20251129_130238.jpg", "20251129_130306.jpg", "20251129_130402.jpg", "20251129_130457.jpg", "20251129_130522.jpg", "20251129_130644.jpg", "20251129_130715.jpg", "20251129_130723.jpg", "20251129_130733.jpg", "20251129_130802.jpg", "20251129_130807.jpg", "20251129_130817.jpg", "20251129_130850.jpg", "20251129_131040.jpg", "20251129_131046.jpg", "20251129_131056.jpg", "20251129_131125.jpg", "20251129_131202.jpg", "20251129_131320.jpg", "20251129_131514.jpg", "20251129_131521.jpg", "20251129_131525.jpg", "20251129_131529.jpg", "20251129_131655.jpg", "20251129_131720.jpg", "20251129_131750.jpg", "20251129_131815.jpg", "20251129_131820.jpg", "20251129_131839.jpg", "20251129_131901.jpg", "20251129_131914.jpg", "20251129_131938.jpg", "20251129_132038.jpg", "20251129_132125.jpg", "20251129_132200.jpg", "20251129_132354.jpg", "20251129_132449.jpg", "20251129_132501.jpg", "20251129_132528.jpg", "20251129_132534.jpg", "20251129_132732.jpg", "20251129_132751.jpg", "20251129_132832.jpg", "20251129_132854.jpg", "20251129_132915.jpg", "20251129_132921.jpg", "20251129_132954.jpg", "20251129_133000.jpg", "20251129_133018.jpg", "20251129_133340.jpg", "20251129_133353.jpg", "20251129_133423.jpg", "20251129_133456.jpg", "20251129_133515.jpg", "20251129_133519.jpg", "20251129_133625.jpg", "20251129_133633.jpg", "20251129_133740.jpg", "20251129_133814.jpg", "20251129_133850.jpg", "20251129_133912.jpg", "20251129_133953.jpg", "20251129_133959.jpg", "20251129_134106.jpg", "20251129_134335.jpg", "20251129_134401.jpg", "20251129_134420.jpg", "20251129_134502.jpg", "20251129_134506.jpg", "20251129_134517.jpg", "20251129_134527.jpg", "20251129_134557.jpg", "20251129_134615.jpg", "20251129_134713.jpg", "20251129_134724.jpg", "20251129_135000.jpg", "20251129_135004.jpg", "20251129_135020.jpg", "20251129_135027.jpg", "20251129_135036.jpg", "20251129_135042.jpg", "20251129_135118.jpg", "20251129_135142.jpg", "20251129_135152.jpg", "20251129_135205.jpg", "20251129_135212.jpg", "20251129_135217.jpg", "20251129_135225.jpg", "20251129_135235.jpg", "20251129_135253.jpg", "20251129_135335.jpg", "20251129_135341.jpg", "20251129_135422.jpg", "20251129_135428.jpg", "20251129_135448.jpg", "20251129_135510.jpg", "20251129_135538.jpg", "20251129_135551.jpg", "20251129_135556.jpg", "20251129_135755.jpg", "20251129_135804.jpg", "20251129_135827.jpg", "20251129_135845.jpg", "20251129_140008.jpg", "20251129_140011.jpg", "20251129_140020.jpg", "20251129_140038.jpg", "20251129_140128.jpg", "20251129_140136.jpg", "20251129_140207.jpg", "20251129_140216.jpg", "20251129_140227.jpg", "20251129_140240.jpg", "20251129_140502.jpg", "20251129_140505.jpg", "20251129_140513.jpg", "20251129_140528.jpg", "20251129_140538.jpg", "20251129_140553.jpg", "20251129_140618.jpg", "20251129_140653.jpg", "20251129_140657.jpg", "20251129_140708.jpg", "20251129_140713.jpg", "20251129_140744.jpg", "20251129_140816.jpg", "20251129_140820.jpg", "20251129_140842.jpg", "20251129_140850.jpg", "20251129_140907.jpg", "20251129_140938.jpg"], "sana": ["Sana 1.jpg", "Sana 100.jpg", "Sana 103.jpg", "Sana 106.jpg", "Sana 107.jpg", "Sana 109.jpg", "Sana 110.jpg", "Sana 111.jpg", "Sana 114.jpg", "Sana 115.jpg", "Sana 118.jpg", "Sana 121.jpg", "Sana 124.jpg", "Sana 125.jpg", "Sana 127.jpg", "Sana 129.jpg", "Sana 131.jpg", "Sana 138.jpg", "Sana 14.jpg", "Sana 141.jpg", "Sana 143.jpg", "Sana 147.jpg", "Sana 148.jpg", "Sana 15.jpg", "Sana 154.jpg", "Sana 156.jpg", "Sana 163.jpg", "Sana 169.jpg", "Sana 173.jpg", "Sana 178.jpg", "Sana 18.jpg", "Sana 183.jpg", "Sana 188.jpg", "Sana 192.jpg", "Sana 194.jpg", "Sana 199.jpg", "Sana 20.jpg", "Sana 201.jpg", "Sana 203.jpg", "Sana 205.jpg", "Sana 206.jpg", "Sana 207.jpg", "Sana 208.jpg", "Sana 214.jpg", "Sana 221.jpg", "Sana 222.jpg", "Sana 223.jpg", "Sana 224.jpg", "Sana 225.jpg", "Sana 227.jpg", "Sana 237.jpg", "Sana 240.jpg", "Sana 241.jpg", "Sana 243.jpg", "Sana 244.jpg", "Sana 245.jpg", "Sana 249.jpg", "Sana 251.jpg", "Sana 253.jpg", "Sana 254.jpg", "Sana 255.jpg", "Sana 258.jpg", "Sana 259.jpg", "Sana 260.jpg", "Sana 261.jpg", "Sana 264.jpg", "Sana 267.jpg", "Sana 268.jpg", "Sana 269.jpg", "Sana 27.jpg", "Sana 272.jpg", "Sana 276.jpg", "Sana 278.jpg", "Sana 288.jpg", "Sana 29.jpg", "Sana 293.jpg", "Sana 294.jpg", "Sana 295.jpg", "Sana 307.jpg", "Sana 311.jpg", "Sana 313.jpg", "Sana 32.jpg", "Sana 323.jpg", "Sana 332.jpg", "Sana 339.jpg", "Sana 34.jpg", "Sana 340.jpg", "Sana 345.jpg", "Sana 346.jpg", "Sana 347.jpg", "Sana 349.jpg", "Sana 35.jpg", "Sana 350.jpg", "Sana 351.jpg", "Sana 36.jpg", "Sana 361.jpg", "Sana 369.jpg", "Sana 37.jpg", "Sana 374.jpg", "Sana 375.jpg", "Sana 377.jpg", "Sana 380.jpg", "Sana 383.jpg", "Sana 384.jpg", "Sana 386.jpg", "Sana 387.jpg", "Sana 388.jpg", "Sana 389.jpg", "Sana 39.jpg", "Sana 392.jpg", "Sana 393.jpg", "Sana 395.jpg", "Sana 399.jpg", "Sana 40.jpg", "Sana 400.jpg", "Sana 401.jpg", "Sana 408.jpg", "Sana 410.jpg", "Sana 411.jpg", "Sana 413.jpg", "Sana 414.jpg", "Sana 419.jpg", "Sana 421.jpg", "Sana 423.jpg", "Sana 425.jpg", "Sana 426.jpg", "Sana 427.jpg", "Sana 43.jpg", "Sana 432.jpg", "Sana 436.jpg", "Sana 438.jpg", "Sana 442.jpg", "Sana 443.jpg", "Sana 446.jpg", "Sana 453.jpg", "Sana 455.jpg", "Sana 457.jpg", "Sana 46.jpg", "Sana 462.jpg", "Sana 464.jpg", "Sana 466.jpg", "Sana 469.jpg", "Sana 473.jpg", "Sana 476.jpg", "Sana 477.jpg", "Sana 48.jpg", "Sana 481.jpg", "Sana 486.jpg", "Sana 488.jpg", "Sana 489.jpg", "Sana 492.jpg", "Sana 496.jpg", "Sana 498.jpg", "Sana 500.jpg", "Sana 507.jpg", "Sana 509.jpg", "Sana 51.jpg", "Sana 511.jpg", "Sana 512.jpg", "Sana 513.jpg", "Sana 514.jpg", "Sana 515.jpg", "Sana 519.jpg", "Sana 52.jpg", "Sana 525.jpg", "Sana 530.jpg", "Sana 531.jpg", "Sana 532.jpg", "Sana 533.jpg", "Sana 536.jpg", "Sana 537.jpg", "Sana 539.jpg", "Sana 548.jpg", "Sana 549.jpg", "Sana 55.jpg", "Sana 551.jpg", "Sana 553.jpg", "Sana 555.jpg", "Sana 556.jpg", "Sana 557.jpg", "Sana 560.jpg", "Sana 565.jpg", "Sana 569.jpg", "Sana 57.jpg", "Sana 572.jpg", "Sana 575.jpg", "Sana 576.jpg", "Sana 577.jpg", "Sana 580.jpg", "Sana 581.jpg", "Sana 582.jpg", "Sana 583.jpg", "Sana 585.jpg", "Sana 586.jpg", "Sana 588.jpg", "Sana 59.jpg", "Sana 591.jpg", "Sana 598.jpg", "Sana 6.jpg", "Sana 60.jpg", "Sana 601.jpg", "Sana 602.jpg", "Sana 608.jpg", "Sana 610.jpg", "Sana 614.jpg", "Sana 617.jpg", "Sana 619.jpg", "Sana 621.jpg", "Sana 622.jpg", "Sana 624.jpg", "Sana 626.jpg", "Sana 629.jpg", "Sana 63.jpg", "Sana 634.jpg", "Sana 637.jpg", "Sana 638.jpg", "Sana 64.jpg", "Sana 643.jpg", "Sana 647.jpg", "Sana 651.jpg", "Sana 653.jpg", "Sana 655.jpg", "Sana 656.jpg", "Sana 657.jpg", "Sana 663.jpg", "Sana 667.jpg", "Sana 668.jpg", "Sana 67.jpg", "Sana 671.jpg", "Sana 672.jpg", "Sana 674.jpg", "Sana 677.jpg", "Sana 682.jpg", "Sana 685.jpg", "Sana 69.jpg", "Sana 691.jpg", "Sana 694.jpg", "Sana 696.jpg", "Sana 7.jpg", "Sana 70.jpg", "Sana 700.jpg", "Sana 701.jpg", "Sana 705.jpg", "Sana 708.jpg", "Sana 715.jpg", "Sana 717.jpg", "Sana 720.jpg", "Sana 723.jpg", "Sana 725.jpg", "Sana 728.jpg", "Sana 73.jpg", "Sana 731.jpg", "Sana 737.jpg", "Sana 739.jpg", "Sana 74.jpg", "Sana 741.jpg", "Sana 743.jpg", "Sana 75.jpg", "Sana 750.jpg", "Sana 752.jpg", "Sana 755.jpg", "Sana 757.jpg", "Sana 762.jpg", "Sana 766.jpg", "Sana 771.jpg", "Sana 775.jpg", "Sana 776.jpg", "Sana 777.jpg", "Sana 779.jpg", "Sana 78.jpg", "Sana 786.jpg", "Sana 788.jpg", "Sana 790.jpg", "Sana 794.jpg", "Sana 795.jpg", "Sana 80.jpg", "Sana 803.jpg", "Sana 808.jpg", "Sana 809.jpg", "Sana 81.jpg", "Sana 813.jpg", "Sana 814.jpg", "Sana 816.jpg", "Sana 817.jpg", "Sana 818.jpg", "Sana 819.jpg", "Sana 820.jpg", "Sana 822.jpg", "Sana 824.jpg", "Sana 827.jpg", "Sana 830.jpg", "Sana 839.jpg", "Sana 84.jpg", "Sana 841.jpg", "Sana 842.jpg", "Sana 843.jpg", "Sana 844.jpg", "Sana 845.jpg", "Sana 846.jpg", "Sana 847.jpg", "Sana 853.jpg", "Sana 857.jpg", "Sana 86.jpg", "Sana 860.jpg", "Sana 861.jpg", "Sana 863.jpg", "Sana 87.jpg", "Sana 873.jpg", "Sana 875.jpg", "Sana 877.jpg", "Sana 886.jpg", "Sana 89.jpg", "Sana 890.jpg", "Sana 892.jpg", "Sana 900.jpg", "Sana 902.jpg", "Sana 903.jpg", "Sana 905.jpg", "Sana 906.jpg", "Sana 91.jpg", "Sana 912.jpg", "Sana 913.jpg", "Sana 917.jpg", "Sana 919.jpg", "Sana 921.jpg", "Sana 928.jpg", "Sana 931.jpg", "Sana 932.jpg", "Sana 934.jpg", "Sana 935.jpg", "Sana 937.jpg", "Sana 938.jpg", "Sana 94.jpg", "Sana 940.jpg", "Sana 943.jpg", "Sana 951.jpg", "Sana 953.jpg", "Sana 956.jpg", "Sana 958.jpg", "Sana 960.jpg", "Sana 966.jpg", "Sana 968.jpg", "Sana 972.jpg", "Sana 976.jpg", "Sana 979.jpg", "Sana 983.jpg", "Sana 99.jpg", "Sana 991.jpg", "Sana 993.jpg"]}, "validation": {"ojo_de_gallo": ["20251129_092009.jpg", "20251129_092333.jpg", "20251129_092735.jpg", "20251129_092925.jpg", "20251129_093509.jpg", "20251129_093848.jpg", "20251129_094057.jpg", "20251129_094345.jpg", "20251129_094716.jpg", "20251129_095022.jpg", "20251129_095346.jpg", "20251129_095547.jpg", "20251129_095626.jpg", "20251129_095703.jpg", "20251129_095949.jpg", "20251129_100037.jpg", "20251129_100540.jpg", "20251129_100648.jpg", "20251129_100822.jpg", "20251129_101002.jpg", "20251129_101038.jpg", "20251129_101044.jpg", "20251129_101243.jpg", "20251129_101733.jpg", "20251129_102732.jpg", "20251129_102831.jpg", "20251129_103110.jpg", "20251129_103945.jpg", "20251129_104403.jpg", "20251129_104738.jpg", "20251129_104907.jpg", "20251129_104918.jpg", "20251129_105430.jpg", "20251129_105445.jpg", "20251129_114552.jpg", "20251129_114708.jpg", "20251129_114714.jpg", "20251129_114939.jpg", "20251129_114946.jpg", "20251129_115257.jpg", "20251129_115342.jpg", "20251129_115513.jpg", "20251129_115618.jpg", "20251129_115807.jpg", "20251129_120332.jpg", "20251129_120401.jpg", "20251129_120633.jpg", "20251129_120920.jpg", "20251129_130229.jpg", "20251129_130232.jpg", "20251129_130322.jpg", "20251129_130533.jpg", "20251129_130657.jpg", "20251129_131129.jpg", "20251129_131356.jpg", "20251129_131402.jpg", "20251129_131638.jpg", "20251129_132216.jpg", "20251129_132305.jpg", "20251129_132632.jpg", "20251129_132910.jpg", "20251129_133030.jpg", "20251129_133527.jpg", "20251129_133840.jpg", "20251129_133933.jpg", "20251129_133943.jpg", "20251129_134656.jpg", "20251129_134708.jpg", "20251129_134753.jpg", "20251129_135137.jpg", "20251129_135531.jpg", "20251129_135606.jpg", "20251129_135624.jpg", "20251129_135927.jpg", "20251129_140749.jpg"], "sana": ["Sana 135.jpg", "Sana 144.jpg", "Sana 155.jpg", "Sana 157.jpg", "Sana 161.jpg", "Sana 19.jpg", "Sana 197.jpg", "Sana 229.jpg", "Sana 239.jpg", "Sana 252.jpg", "Sana 257.jpg", "Sana 297.jpg", "Sana 300.jpg", "Sana 314.jpg", "Sana 316.jpg", "Sana 342.jpg", "Sana 352.jpg", "Sana 355.jpg", "Sana 356.jpg", "Sana 359.jpg", "Sana 379.jpg", "Sana 398.jpg", "Sana 403.jpg", "Sana 428.jpg", "Sana 435.jpg", "Sana 441.jpg", "Sana 467.jpg", "Sana 471.jpg", "Sana 472.jpg", "Sana 50.jpg", "Sana 524.jpg", "Sana 526.jpg", "Sana 528.jpg", "Sana 558.jpg", "Sana 567.jpg", "Sana 574.jpg", "Sana 592.jpg", "Sana 596.jpg", "Sana 611.jpg", "Sana 62.jpg", "Sana 632.jpg", "Sana 633.jpg", "Sana 652.jpg", "Sana 675.jpg", "Sana 684.jpg", "Sana 687.jpg", "Sana 693.jpg", "Sana 697.jpg", "Sana 703.jpg", "Sana 718.jpg", "Sana 72.jpg", "Sana 721.jpg", "Sana 749.jpg", "Sana 765.jpg", "Sana 781.jpg", "Sana 805.jpg", "Sana 826.jpg", "Sana 828.jpg", "Sana 856.jpg", "Sana 866.jpg", "Sana 876.jpg", "Sana 879.jpg", "Sana 882.jpg", "Sana 897.jpg", "Sana 910.jpg", "Sana 936.jpg", "Sana 939.jpg", "Sana 950.jpg", "Sana 957.jpg", "Sana 962.jpg", "Sana 974.jpg", "Sana 980.jpg", "Sana 987.jpg", "Sana 994.jpg"]}, "test": {"ojo_de_gallo": ["20251129_091427.jpg", "20251129_092633.jpg", "20251129_092724.jpg", "20251129_093405.jpg", "20251129_094358.jpg", "20251129_094634.jpg", "20251129_094823.jpg", "20251129_094932.jpg", "20251129_095649.jpg", "20251129_095825.jpg", "20251129_095836.jpg", "20251129_100024.jpg", "20251129_100529.jpg", "20251129_101314.jpg", "20251129_101826.jpg", "20251129_101959.jpg", "20251129_102055.jpg", "20251129_102103.jpg", "20251129_102113.jpg", "20251129_102445.jpg", "20251129_102558.jpg", "20251129_102920.jpg", "20251129_103018.jpg", "20251129_103228.jpg", "20251129_103610.jpg", "20251129_103921.jpg", "20251129_104056.jpg", "20251129_104456.jpg", "20251129_104554.jpg", "20251129_104647.jpg", "20251129_104818.jpg", "20251129_104945.jpg", "20251129_105017.jpg", "20251129_105036.jpg", "20251129_115401.jpg", "20251129_115438.jpg", "20251129_115623.jpg", "20251129_115857.jpg", "20251129_120211.jpg", "20251129_120253.jpg", "20251129_120538.jpg", "20251129_120659.jpg", "20251129_120717.jpg", "20251129_120953.jpg", "20251129_121103.jpg", "20251129_130241.jpg", "20251129_130419.jpg", "20251129_131111.jpg", "20251129_131229.jpg", "20251129_131241.jpg", "20251129_131303.jpg", "20251129_131351.jpg", "20251129_131423.jpg", "20251129_131620.jpg", "20251129_131753.jpg", "20251129_131923.jpg", "20251129_132248.jpg", "20251129_132523.jpg", "20251129_132727.jpg", "20251129_132740.jpg", "20251129_132859.jpg", "20251129_133824.jpg", "20251129_134344.jpg", "20251129_134917.jpg", "20251129_135125.jpg", "20251129_135900.jpg", "20251129_140025.jpg", "20251129_140043.jpg", "20251129_140236.jpg", "20251129_140508.jpg", "20251129_140516.jpg", "20251129_140753.jpg", "20251129_140759.jpg", "20251129_140802.jpg", "20251129_140813.jpg"], "sana": ["Sana 104.jpg", "Sana 119.jpg", "Sana 12.jpg", "Sana 122.jpg", "Sana 123.jpg", "Sana 133.jpg", "Sana 177.jpg", "Sana 187.jpg", "Sana 191.jpg", "Sana 193.jpg", "Sana 198.jpg", "Sana 204.jpg", "Sana 215.jpg", "Sana 23.jpg", "Sana 248.jpg", "Sana 250.jpg", "Sana 275.jpg", "Sana 279.jpg", "Sana 308.jpg", "Sana 309.jpg", "Sana 31.jpg", "Sana 312.jpg", "Sana 320.jpg", "Sana 341.jpg", "Sana 354.jpg", "Sana 360.jpg", "Sana 365.jpg", "Sana 382.jpg", "Sana 417.jpg", "Sana 420.jpg", "Sana 422.jpg", "Sana 431.jpg", "Sana 433.jpg", "Sana 45.jpg", "Sana 452.jpg", "Sana 490.jpg", "Sana 494.jpg", "Sana 516.jpg", "Sana 527.jpg", "Sana 571.jpg", "Sana 594.jpg", "Sana 600.jpg", "Sana 603.jpg", "Sana 615.jpg", "Sana 627.jpg", "Sana 635.jpg", "Sana 636.jpg", "Sana 649.jpg", "Sana 650.jpg", "Sana 660.jpg", "Sana 670.jpg", "Sana 68.jpg", "Sana 689.jpg", "Sana 711.jpg", "Sana 730.jpg", "Sana 732.jpg", "Sana 735.jpg", "Sana 745.jpg", "Sana 761.jpg", "Sana 77.jpg", "Sana 774.jpg", "Sana 79.jpg", "Sana 792.jpg", "Sana 832.jpg", "Sana 837.jpg", "Sana 838.jpg", "Sana 850.jpg", "Sana 869.jpg", "Sana 894.jpg", "Sana 914.jpg", "Sana 933.jpg", "Sana 959.jpg", "Sana 965.jpg", "Sana 97.jpg", "Sana 977.jpg", "Sana 990.jpg"]}}''')
ROYA_M = listar(RUTAS["roya"]); div_r = dividir(len(ROYA_M))
MEN = {}
for p in ["train", "validation", "test"]:
    MEN[("ojo_de_gallo", p)] = leer_varias([os.path.join(RUTAS["ojo_de_gallo"], f) for f in PARTICION[p]["ojo_de_gallo"]], f"Mendeley ojo {p}")[0]
    MEN[("sana", p)] = leer_varias([os.path.join(RUTAS["sana"], f) for f in PARTICION[p]["sana"]], f"Mendeley sana {p}")[0]
    MEN[("roya", p)] = leer_varias([ROYA_M[i] for i in div_r[p]], f"Mendeley roya {p}")[0]

def leer_tabla(df, desc):
    """Divide 70/15/15 por clase (semilla 42) y lee las imágenes."""
    out = {}
    for c in sorted(df["clase"].unique()):
        rutas = sorted(df.loc[df["clase"] == c, "ruta"].tolist()); d = dividir(len(rutas))
        for p, ii in d.items():
            out[(c, p)] = leer_varias([rutas[i] for i in ii], f"{desc} {c} {p}")
    return out
ROC = leer_tabla(ANOT, "RoCoLe")
BRA = leer_tabla(TB, "BRACOL")
pd.DataFrame([{"conjunto": n, "clase": c, "particion": p, "imagen": os.path.basename(r)}
              for n, D in [("RoCoLe", ROC), ("BRACOL", BRA)] for (c, p), (_, rs) in D.items() for r in rs]
             ).to_csv(os.path.join(SAL, "tablas", "particion_roya.csv"), index=False)

PG = {}
for carpeta, clase in [("ojo", "ojo_de_gallo"), ("roya", "roya"), ("healthy", "sana")]:
    X, rutas = leer_varias(listar(os.path.join(RUTAS["pg26038"], carpeta)), f"PG {carpeta}")
    PG[clase] = (X, [os.path.basename(r).split("__")[0] for r in rutas], [os.path.basename(r) for r in rutas])

Mendeley ojo train: 350 de 350 en 111 s
Mendeley sana train: 349 de 349 en 98 s
Mendeley roya train: 350 de 350 en 109 s
Mendeley ojo validation: 75 de 75 en 21 s
Mendeley sana validation: 74 de 74 en 20 s
Mendeley roya validation: 75 de 75 en 27 s
Mendeley ojo test: 75 de 75 en 23 s
Mendeley sana test: 76 de 76 en 23 s
Mendeley roya test: 74 de 75 en 23 s
RoCoLe otra train: 117 de 117 en 16 s
RoCoLe otra validation: 25 de 25 en 4 s
RoCoLe otra test: 25 de 25 en 4 s
RoCoLe roya train: 421 de 421 en 30 s
RoCoLe roya validation: 90 de 90 en 7 s
RoCoLe roya test: 91 de 91 en 6 s
RoCoLe sana train: 554 de 554 en 45 s
RoCoLe sana validation: 119 de 119 en 9 s
RoCoLe sana test: 118 de 118 en 10 s
BRACOL no_roya train: 557 de 558 en 43 s
BRACOL no_roya validation: 120 de 120 en 10 s
BRACOL no_roya test: 119 de 119 en 10 s
BRACOL roya train: 424 de 424 en 36 s
BRACOL roya validation: 91 de 91 en 8 s
BRACOL roya test: 90 de 90 en 9 s
PG ojo: 40 de 40 en 5 s
PG roya: 40 de 40 en 3 s
PG healthy: 

## 5. Conjuntos del detector de roya
Etiqueta 1 = roya; 0 = no es roya (ojo de gallo, sana u otra mancha).

In [6]:
def partes(p):
    """Lista de (origen, clase original, imágenes, etiqueta)."""
    L = [("Mendeley", "roya", MEN[("roya", p)], 1), ("Mendeley", "ojo_de_gallo", MEN[("ojo_de_gallo", p)], 0),
         ("Mendeley", "sana", MEN[("sana", p)], 0)]
    for (c, q), (X, _) in ROC.items():
        if q == p: L.append(("RoCoLe", c, X, int(c == "roya")))
    for (c, q), (X, _) in BRA.items():
        if q == p: L.append(("BRACOL", c, X, int(c == "roya")))
    return [t for t in L if len(t[2])]
def conjunto(p):
    L = partes(p)
    return np.concatenate([t[2] for t in L]), np.concatenate([np.full(len(t[2]), t[3]) for t in L]).astype("float32")
Xtr, ytr = conjunto("train"); Xva, yva = conjunto("validation")
conteo = pd.DataFrame([{"particion": p, "origen": o, "clase": c, "n": len(X), "roya": e}
                       for p in ["train", "validation", "test"] for o, c, X, e in partes(p)])
print(conteo.pivot_table(index=["origen", "clase"], columns="particion", values="n", aggfunc="sum"))
conteo.to_csv(os.path.join(SAL, "tablas", "conteo_conjuntos.csv"), index=False)
pesos = {0: len(ytr) / (2 * (ytr == 0).sum()), 1: len(ytr) / (2 * (ytr == 1).sum())}
print("Entrenamiento:", len(ytr), "roya:", int(ytr.sum()), "| validación:", len(yva), "| pesos:", pesos)

particion              test  train  validation
origen   clase                                
BRACOL   no_roya        119    557         120
         roya            90    424          91
Mendeley ojo_de_gallo    75    350          75
         roya            74    350          75
         sana            76    349          74
RoCoLe   otra            25    117          25
         roya            91    421          90
         sana           118    554         119
Entrenamiento: 3122 roya: 1195 | validación: 669 | pesos: {0: np.float64(0.8100674623767514), 1: np.float64(1.306276150627615)}


## 6. Modelo: MobileNetV3Small (base congelada), misma receta de las fases anteriores
Aumento de datos: giros, rotación, zoom, contraste y brillo. Adam 1e-3; se conserva la época con menor pérdida de
validación.

In [7]:
fijar_semillas()
e = keras.Input((TAM, TAM, 3), name="imagen")
aum = keras.Sequential([keras.layers.RandomFlip("horizontal_and_vertical"), keras.layers.RandomRotation(0.25),
                        keras.layers.RandomZoom(0.15), keras.layers.RandomContrast(0.2),
                        keras.layers.RandomBrightness(0.2, value_range=(0, 255))], name="data_augmentation")
base = keras.applications.MobileNetV3Small(input_shape=(TAM, TAM, 3), include_top=False,
                                           weights=None if MOCK else "imagenet", include_preprocessing=True)
base.trainable = False
z = base(aum(e), training=False)
z = keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(z)
z = keras.layers.Dropout(0.2, name="dropout")(z)
z = keras.layers.Dense(1, activation="sigmoid", name="es_roya")(z)
modelo = keras.Model(e, z, name="detector_roya")
modelo.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
               metrics=["accuracy", keras.metrics.AUC(name="auc")])
mejor = os.path.join(TMP, "roya_mejor.weights.h5")
h = modelo.fit(ds_desde(Xtr, ytr, True), validation_data=ds_desde(Xva, yva, False), epochs=EPOCAS,
               class_weight=pesos, verbose=2,
               callbacks=[keras.callbacks.ModelCheckpoint(mejor, monitor="val_loss", save_best_only=True, save_weights_only=True)])
modelo.load_weights(mejor)
guardar_json({k: [float(v) for v in vs] for k, vs in h.history.items()}, "historial_entrenamiento.json")
del Xtr; import gc; gc.collect()

4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/20
98/98 - 17s - 174ms/step - accuracy: 0.5778 - auc: 0.6179 - loss: 0.6765 - val_accuracy: 0.7399 - val_auc: 0.8097 - val_loss: 0.5532
Epoch 2/20
98/98 - 4s - 37ms/step - accuracy: 0.6986 - auc: 0.7609 - loss: 0.5869 - val_accuracy: 0.7235 - val_auc: 0.8386 - val_loss: 0.5591
Epoch 3/20
98/98 - 5s - 46ms/step - accuracy: 0.7309 - auc: 0.7990 - loss: 0.5533 - val_accuracy: 0.7833 - val_auc: 0.8638 - val_loss: 0.5016
Epoch 4/20
98/98 - 4s - 41ms/step - accuracy: 0.7562 - auc: 0.8248 - loss: 0.5246 - val_accuracy: 0.7997 - val_auc: 0.8755 - val_loss: 0.4767
Epoch 5/20
98/98 - 5s - 51ms/step - accuracy: 0.7578 - auc: 0.8328 - loss: 0.5121 - val_accuracy: 0.7967 - val_auc: 0.8835 - val_loss: 0.4703
Epoch 6/20
98/98 - 4s - 39ms/step - accuracy: 0.7585 - auc: 0.8404 - loss: 0.5015 - val_accuracy: 0.8161 - val_auc: 0.8912 - val_loss: 0.4407
Epoch 7/20
98/98 - 4s - 40ms/step - accuracy: 0.7748 - auc: 0.8517 - loss: 0.4873 - val_accuracy:

1681

## 7. Conversión a TensorFlow Lite (float16) y modelos de la aplicación

In [8]:
def inferencia(m):
    ent = keras.Input((TAM, TAM, 3), name="imagen"); x = ent
    for capa in m.layers[1:]:
        if capa.name == "data_augmentation": continue
        x = capa(x, training=False)
    return keras.Model(ent, x)
def a_tflite(m, nombre):
    carpeta = os.path.join(TMP, "sm_" + nombre)
    try: m.export(carpeta, verbose=False)
    except TypeError: m.export(carpeta)
    c = tf.lite.TFLiteConverter.from_saved_model(carpeta)
    c.optimizations = [tf.lite.Optimize.DEFAULT]; c.target_spec.supported_types = [tf.float16]
    return c.convert()
class TFL:
    def __init__(self, contenido):
        self.it = tf.lite.Interpreter(model_content=contenido); self.it.allocate_tensors()
        self.i = self.it.get_input_details()[0]; self.o = self.it.get_output_details()[0]
    def __call__(self, X):
        out = []
        for k in range(len(X)):
            self.it.set_tensor(self.i["index"], X[k:k + 1].astype("float32")); self.it.invoke()
            out.append(float(self.it.get_tensor(self.o["index"]).ravel()[0]))
        return np.array(out)
def vistas(X):
    return [X, X[:, :, ::-1], X[:, ::-1, :], X[:, ::-1, ::-1]]      # original, espejo horizontal, vertical, 180°
def tta(f, X):
    return np.mean([f(np.ascontiguousarray(v)) for v in vistas(X)], axis=0)

modelo_inf = inferencia(modelo)
B_ROYA = a_tflite(modelo_inf, "roya")
ROYA_T = TFL(B_ROYA)
ref = modelo_inf.predict(Xva.astype("float32"), verbose=0).ravel(); tfl = ROYA_T(Xva)
conv = {"acuerdo_decision": float(((ref >= .5) == (tfl >= .5)).mean()), "dif_max": float(np.abs(ref - tfl).max()),
        "MB": len(B_ROYA) / 1e6, "salida": "P(roya)"}
print(conv); guardar_json(conv, "conversion_tflite.json")
with open(os.path.join(SAL, "android", "modelo_roya.tflite"), "wb") as f: f.write(B_ROYA)
DIAG = TFL(open(RUTAS["diag_v1"], "rb").read()); VER = TFL(open(RUTAS["ver_v3"], "rb").read())
def p_ojo_tta(X): return 1.0 - tta(DIAG, X)         # el v1 entrega P(no es ojo de gallo)

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


{'acuerdo_decision': 1.0, 'dif_max': 0.01481318473815918, 'MB': 1.943576, 'salida': 'P(roya)'}


## 8. Regla de color (comparación)
Proporción de píxeles anaranjados (tono 18°–48°, saturación ≥ 0,55, brillo ≥ 0,45) entre los píxeles de hoja.
El umbral se elige con la validación (máximo índice de Youden), igual que el del modelo.

In [9]:
def color_naranja(X):
    out = []
    for x in X:
        hsv = np.asarray(Image.fromarray(x).convert("HSV"), dtype=np.float32) / 255.0
        h, s, v = hsv[..., 0] * 360, hsv[..., 1], hsv[..., 2]
        nar = (h >= 18) & (h <= 48) & (s >= .55) & (v >= .45)
        hoja = ((h >= 60) & (h <= 170) & (s >= .15)) | nar
        out.append(nar.sum() / max(hoja.sum(), 1))
    return np.array(out)
def umbral_youden(y, p):
    fpr, tpr, thr = roc_curve(y, p); i = int(np.argmax(tpr - fpr)); return float(min(max(thr[i], 1e-6), 1.0))
pva_modelo = tta(ROYA_T, Xva); pva_color = color_naranja(Xva)
U_MODELO = umbral_youden(yva, pva_modelo); U_COLOR = umbral_youden(yva, pva_color)
print(f"Umbral del modelo (validación): {U_MODELO:.3f} | umbral de color: {U_COLOR:.4f}")
print(f"AUC validación: modelo {roc_auc_score(yva, pva_modelo):.3f} | color {roc_auc_score(yva, pva_color):.3f}")
guardar_json({"umbral_modelo": U_MODELO, "umbral_color": U_COLOR,
              "auc_val_modelo": roc_auc_score(yva, pva_modelo), "auc_val_color": roc_auc_score(yva, pva_color)}, "umbrales.json")
del Xva; gc.collect()

Umbral del modelo (validación): 0.444 | umbral de color: 0.0004
AUC validación: modelo 0.917 | color 0.737


10758

## 9. Resultados por conjunto (prueba de Mendeley, RoCoLe y BRACOL; PG 26038 externo)
Para cada grupo: cuántas imágenes se marcan como roya, con intervalo de Wilson del 95 %.

In [10]:
grupos = [(f"{o} {c} (prueba)", X, e) for o, c, X, e in partes("test")]
grupos += [(f"PG 26038 {c} (externo)", PG[c][0], int(c == "roya")) for c in ["ojo_de_gallo", "roya", "sana"]]
filas, PRED = [], {}
for nombre, X, e in grupos:
    pm = tta(ROYA_T, X); pc = color_naranja(X); PRED[nombre] = (pm, pc)
    for metodo, p, u in [("modelo", pm, U_MODELO), ("color", pc, U_COLOR)]:
        k = int((p >= u).sum()); n = len(p)
        filas.append({"grupo": nombre, "metodo": metodo, "es_roya": e, "n": n, "marcadas_roya": k,
                      "tasa": k / n, "ic95_inf": wilson(k, n)[0], "ic95_sup": wilson(k, n)[1],
                      "acierto": (k if e else n - k) / n})
T = pd.DataFrame(filas); T.to_csv(os.path.join(SAL, "tablas", "deteccion_roya_por_grupo.csv"), index=False)
pd.set_option("display.width", 200)
print(T.pivot_table(index="grupo", columns="metodo", values="tasa").round(3))
for metodo in ["modelo", "color"]:
    pg = T[(T.metodo == metodo) & T.grupo.str.startswith("PG")]
    print(metodo, "PG:", dict(zip(pg.grupo, pg.tasa.round(3))))

metodo                           color  modelo
grupo                                         
BRACOL no_roya (prueba)          0.378   0.151
BRACOL roya (prueba)             0.878   0.833
Mendeley ojo_de_gallo (prueba)   0.920   0.000
Mendeley roya (prueba)           0.932   0.838
Mendeley sana (prueba)           0.026   0.092
PG 26038 ojo_de_gallo (externo)  0.025   0.100
PG 26038 roya (externo)          0.025   0.000
PG 26038 sana (externo)          0.000   0.000
RoCoLe otra (prueba)             0.840   0.720
RoCoLe roya (prueba)             0.626   0.758
RoCoLe sana (prueba)             0.271   0.297
modelo PG: {'PG 26038 ojo_de_gallo (externo)': 0.1, 'PG 26038 roya (externo)': 0.0, 'PG 26038 sana (externo)': 0.0}
color PG: {'PG 26038 ojo_de_gallo (externo)': 0.025, 'PG 26038 roya (externo)': 0.025, 'PG 26038 sana (externo)': 0.0}


## 10. Regla completa de la aplicación en PG 26038 (externo) y en la prueba de Mendeley
- **Actual:** verificador v3 → diagnóstico v1 con promedio de 4 vistas → bandas 0,30 / 0,70.
- **Nueva:** igual, pero si la hoja tiene lesiones (P(ojo) ≥ 0,30) y el detector dice roya (P(roya) ≥ umbral),
  el resultado es **"parece roya"**.

In [11]:
def decidir(ph, po, pr, usar_roya, u):
    if ph < 0.5: return "no_valida"
    if usar_roya and po >= 0.30 and pr >= u: return "roya"
    if po >= 0.70: return "ojo_de_gallo"
    if po <= 0.30: return "sana"
    return "no_concluyente"
evaluar = [("PG 26038", c, PG[c][0]) for c in ["ojo_de_gallo", "roya", "sana"]]
evaluar += [("Mendeley prueba", c, MEN[(c, "test")]) for c in ["ojo_de_gallo", "roya", "sana"]]
filas, detalle = [], []
for origen, clase, X in evaluar:
    ph = VER(X); po = p_ojo_tta(X); pr = tta(ROYA_T, X)          # el verificador se usa sin promedio, como en la app
    for regla, usar in [("actual", False), ("nueva", True)]:
        res = [decidir(a, b, c, usar, U_MODELO) for a, b, c in zip(ph, po, pr)]
        cnt = pd.Series(res).value_counts()
        fila = {"origen": origen, "clase_real": clase, "regla": regla, "n": len(res)}
        for r in ["ojo_de_gallo", "roya", "sana", "no_concluyente", "no_valida"]: fila[r] = int(cnt.get(r, 0))
        filas.append(fila)
    for i in range(len(X)):
        detalle.append({"origen": origen, "clase_real": clase, "p_hoja": ph[i], "p_ojo_tta": po[i], "p_roya_tta": pr[i],
                        "actual": decidir(ph[i], po[i], pr[i], False, U_MODELO), "nueva": decidir(ph[i], po[i], pr[i], True, U_MODELO)})
R = pd.DataFrame(filas); R.to_csv(os.path.join(SAL, "tablas", "regla_completa.csv"), index=False)
pd.DataFrame(detalle).to_csv(os.path.join(SAL, "tablas", "regla_completa_por_imagen.csv"), index=False)
print(R.to_string(index=False))

def resumen(origen):
    out = {}
    for regla in ["actual", "nueva"]:
        r = R[(R.origen == origen) & (R.regla == regla)].set_index("clase_real")
        n_ojo, n_roya, n_sana = r.loc["ojo_de_gallo", "n"], r.loc["roya", "n"], r.loc["sana", "n"]
        out[regla] = {"ojo_detectado": r.loc["ojo_de_gallo", "ojo_de_gallo"] / n_ojo,
                      "ojo_confundido_con_roya": r.loc["ojo_de_gallo", "roya"] / n_ojo,
                      "roya_dicha_ojo": r.loc["roya", "ojo_de_gallo"] / n_roya,
                      "roya_detectada": r.loc["roya", "roya"] / n_roya,
                      "sana_dicha_ojo": r.loc["sana", "ojo_de_gallo"] / n_sana}
    return out
RES = {o: resumen(o) for o in ["PG 26038", "Mendeley prueba"]}
print(json.dumps(RES, indent=2, ensure_ascii=False, default=float))
guardar_json({"versiones": VERSIONES, "umbral_modelo": U_MODELO, "umbral_color": U_COLOR, "conversion": conv,
              "regla_completa": RES, "danadas": DANADAS}, "resumen_fase2e.json")
print("\nArchivos en:", SAL)
print("Modelo para la app:", os.path.join(SAL, "android", "modelo_roya.tflite"))

         origen   clase_real  regla  n  ojo_de_gallo  roya  sana  no_concluyente  no_valida
       PG 26038 ojo_de_gallo actual 40            14     0     3              13         10
       PG 26038 ojo_de_gallo  nueva 40            11     4     3              12         10
       PG 26038         roya actual 40             3     0    21              15          1
       PG 26038         roya  nueva 40             3     0    21              15          1
       PG 26038         sana actual 20             0     0    14               6          0
       PG 26038         sana  nueva 20             0     0    14               6          0
Mendeley prueba ojo_de_gallo actual 75            74     0     0               1          0
Mendeley prueba ojo_de_gallo  nueva 75            74     0     0               1          0
Mendeley prueba         roya actual 74            25     0    37              12          0
Mendeley prueba         roya  nueva 74             0    37    37               0